# Machine RUL Prediction — NASA C-MAPSS FD001

**Author:** Pallavi

## Problem & Data Understanding

Machine failure can lead to unexpected downtime, maintenance costs, and operational disruption.

In this project, I will use the NASA C-MAPSS FD001 dataset to predict the **Remaining Useful Life (RUL)** of simulated turbofan engines using their operating and sensor measurements.

This notebook focuses on understanding the problem and the structure of the dataset before moving into exploratory data analysis and model development.

## 1. Why This Problem?

Predictive maintenance aims to identify when a machine may need attention before failure occurs.

RUL prediction provides an estimate of how many operating cycles a machine has remaining, which can support better maintenance planning.

I chose this problem because it gives us a practical regression problem where machine condition can be connected to a continuous target.

## 2. What Are We Predicting?

The target of this project is **Remaining Useful Life (RUL)**.

RUL represents the number of operating cycles remaining before a machine reaches failure.

For example, if an engine fails at cycle 200 and is currently at cycle 150:

**RUL = 200 − 150 = 50 cycles**

Our goal is to learn the relationship between machine condition and its remaining useful life.

In [1]:
# Importing required libraries

import pandas as pd
import numpy as np

## 3. Loading the Dataset

Before inspecting the data, I will define the paths to the raw FD001 files. Keeping the original data inside `data/raw` helps us separate the source data from any processed data created later.

In [2]:
# Defining the raw dataset paths

train_path = "../data/raw/train_FD001.txt"
test_path = "../data/raw/test_FD001.txt"
rul_path = "../data/raw/RUL_FD001.txt"

## 4. Loading the Training Data

The raw FD001 file does not contain column names. I will define the column structure first and then load the training data so that each measurement can be identified clearly.

In [3]:
# Defining the column names

columns = (
    ["unit_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

len(columns)

26

## 4. Loading the Training Data

I will now load the training data using the column names defined above.

In [4]:
# Loading the training data

train_df = pd.read_csv(
    train_path,
    sep=r"\s+",
    header=None,
    names=columns
)

train_df.head()

,unit_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## 5. Initial Dataset Inspection

Now that the data is loaded, I want to check its size, structure, missing values, and number of engines before going deeper into the analysis.

In [5]:
# Inspecting the dataset

print("Shape:", train_df.shape)
print("\nData types:")
print(train_df.dtypes)
print("\nMissing values:", train_df.isnull().sum().sum())
print("Number of engines:", train_df["unit_id"].nunique())

Shape: (20631, 26)

Data types:
unit_id        int64
cycle          int64
setting_1    float64
setting_2    float64
setting_3    float64
sensor_1     float64
sensor_2     float64
sensor_3     float64
sensor_4     float64
sensor_5     float64
sensor_6     float64
sensor_7     float64
sensor_8     float64
sensor_9     float64
sensor_10    float64
sensor_11    float64
sensor_12    float64
sensor_13    float64
sensor_14    float64
sensor_15    float64
sensor_16    float64
sensor_17      int64
sensor_18      int64
sensor_19    float64
sensor_20    float64
sensor_21    float64
dtype: object

Missing values: 0
Number of engines: 100


## 6. Understanding the Data Range

I will review the basic statistics and engine lifetimes to understand the range of the measurements and how differently the engines operate before failure.

In [7]:
# Reviewing basic statistics
train_df.describe().T
# Checking engine lifetimes
train_df.groupby("unit_id")["cycle"].max().describe()

count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: cycle, dtype: float64

## 7. Checking Constant Features

Some measurements may remain unchanged throughout the dataset. I will identify them before deciding whether they provide useful information for the model.

In [10]:
# Finding constant columns
constant_columns = train_df.columns[train_df.nunique() == 1]
constant_columns
# Counting constant columns
len(constant_columns)

7

## 8. What I Learned From the Dataset

The FD001 training data contains 20,631 observations from 100 simulated engines. Each observation represents one engine at a particular operating cycle and contains 26 columns: the engine identifier, cycle, three operating settings, and 21 sensor measurements.

The engines have different operating lifetimes, ranging from 128 to 362 cycles, with an average lifetime of about 206 cycles. This confirms that each engine follows its own degradation trajectory.

The dataset contains no missing values, while 7 columns have constant values across the training data. I will investigate these columns further before deciding whether they should be removed during preprocessing.

With the basic structure understood, the next step is to explore the data and investigate how the sensor measurements change with machine degradation and RUL.